# MINT-FS / HINT-Net — validation notebook (Steps 1–3)

Runs end to end and writes the paper tables to `results/`:

1. **Step 1** – reproduce the synthetic benchmarks (multi-class `MC`, binary `BIN`)
2. **Step 2** – run the three real datasets (WUSTL-HDRL-2024, ECU-IoHT, WUSTL-EHMS-2020)
3. **Step 3** – repeat every experiment over **10 stratified splits** and report **mean ± s.d.**

The code is CPU-only (a GPU gives no speed-up). Run **one dataset per session** (set `RUN` in the settings cell). Finished datasets are cached in `results/*.json`; the export cell refuses to run with `QUICK = True`.
Whatever numbers this notebook produces are the numbers to put in the paper — do not reuse older ones.

## 0. Setup

In [ ]:
# ---- Option A: upload mintfs_github_repo.zip via the Colab file panel, then run this cell ----
# ---- Option B: set REPO_URL to your GitHub repo and it will be cloned ----
import os, sys, glob, zipfile, subprocess
REPO_URL = ""            # e.g. "https://github.com/<user>/mintfs"
CODE_DIR = "mintfs_code"
if not os.path.exists(os.path.join(CODE_DIR, "mintfs.py")):
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE_DIR], check=True)
    else:
        z = (glob.glob("*mintfs*.zip") + glob.glob("/content/*mintfs*.zip"))
        assert z, "Upload mintfs_github_repo.zip (left panel -> Files) or set REPO_URL"
        os.makedirs(CODE_DIR, exist_ok=True)
        zipfile.ZipFile(z[0]).extractall(CODE_DIR)
    # flatten if the zip contains a single top-level folder
    if not os.path.exists(os.path.join(CODE_DIR, "mintfs.py")):
        sub = glob.glob(os.path.join(CODE_DIR, "*", "mintfs.py"))[0]
        CODE_DIR = os.path.dirname(sub)
sys.path.insert(0, CODE_DIR)
print("code dir:", CODE_DIR, os.listdir(CODE_DIR))

In [ ]:
import numpy as np, pandas as pd, torch, time, json, itertools, warnings
warnings.filterwarnings("ignore")
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier
from mintfs import *
from data import make_synthetic, split_standardise, COUNTS_MC
from prep_real import LOADERS
from config import CFG
torch.set_num_threads(os.cpu_count() or 2)
os.makedirs("results", exist_ok=True)

# ------------------------------ settings ------------------------------
QUICK     = False     # True = smoke test (tiny epochs, 2 seeds; numbers are NOT usable). False = real run.
N_SEEDS   = 5         # number of stratified splits (use 10 for the final paper if time allows)
MAX_ROWS  = 20000     # stratified cap for large real datasets (all rare-class rows are kept); None = no cap
RUN       = ["ECU-IoHT"]   # which real datasets to run in THIS session, e.g. ["WUSTL-EHMS-2020"] or all three
RUN_SYNTHETIC = False     # synthetic benchmarks are optional (slow)
cfg = dict(CFG)
if QUICK:
    cfg.update(epochs=2, epochs1=3, warm=1, ramp=1, spe=5); N_SEEDS = 2; MAX_ROWS = 6000
SEEDS = list(range(N_SEEDS))
print("QUICK =", QUICK, "| seeds:", SEEDS)

## 1. Core runner (one dataset, one split → all methods)

In [ ]:
def fit_eval(S, tr, C, rare, prior, seed, with_rf=True):
    """Retrain HINT-Net (and a balanced RF) on the feature subset S and score on the test split."""
    Xtr, ytr, Xva, yva, Xte, yte = tr
    S = np.array(sorted(int(i) for i in S))
    m = train_classifier(Xtr[:, S], ytr, Xva[:, S], yva, C, cfg, seed=seed)
    tau = tune_logit_adjust(m, Xva[:, S], yva, C, prior)
    pt = predict_proba(m, torch.tensor(Xte[:, S]))
    out = {"HINT-Net": evaluate(yte, pt, C, rare, prior=prior, tau=tau)}
    if with_rf:
        rf = RandomForestClassifier(200, class_weight="balanced_subsample", min_samples_leaf=2,
                                    n_jobs=-1, random_state=seed).fit(Xtr[:, S], ytr)
        out["RF"] = evaluate(yte, rf.predict_proba(Xte[:, S]), C, rare)
    return out

def run_split(X, y, C, seed, lam_sparse, info=None):
    """Select features with MINT-FS and every baseline at the same k; evaluate all on the same split."""
    cfg.update(lam_sparse=lam_sparse)
    tr = split_standardise(X, y, seed)
    Xtr, ytr, Xva, yva, Xte, yte = tr
    counts = np.bincount(ytr, minlength=C); prior = counts / counts.sum(); rare = int(np.argmin(counts))
    rows, sets = [], {}
    def log(method, S, sel_time, res):
        for clf_name, m in res.items():
            rows.append(dict(seed=seed, method=method, clf=clf_name, k=len(S), sel_time=sel_time, **m))
        sets[method] = sorted(int(i) for i in S)
    t = time.time()
    sel, clf, gdet = train_mintfs(Xtr, ytr, Xva, yva, C, cfg, seed=seed)
    S, g = select_subset(clf, gdet, Xva, yva, C); st = time.time() - t
    log("MINT-FS", S, st, fit_eval(S, tr, C, rare, prior, seed)); k = len(S)
    log("No-FS", range(X.shape[1]), 0.0, fit_eval(range(X.shape[1]), tr, C, rare, prior, seed))
    t = time.time(); ranks = baseline_rankings(Xtr, ytr, seed); bt = (time.time() - t) / max(len(ranks), 1)
    for name, rk in ranks.items():
        log(name, rk[:k], bt, fit_eval(rk[:k], tr, C, rare, prior, seed))
    meta = dict(seed=seed, counts=counts.tolist(), rare=rare, d=X.shape[1], k=k, gates=g.tolist())
    if info is not None:   # ground truth exists only for the synthetic data
        Sset = set(sets["MINT-FS"])
        meta["quality"] = dict(n_info=len(Sset & set(info["informative"])), n_noise=len(Sset & set(info["noise"])),
                               n_red=len(Sset & set(info["redundant"])), synergy=bool(set(info["synergy"]) <= Sset))
    return rows, sets, meta

def run_dataset(name, X, y, C, lam_sparse, info=None, feat_names=None):
    path = f"results/{name}.json"
    if os.path.exists(path) and not QUICK:
        print("loading cached", path); return json.load(open(path))
    allrows, allsets, metas = [], [], []
    for s in SEEDS:
        t = time.time(); r, st, m = run_split(X, y, C, s, lam_sparse, info)
        allrows += r; allsets.append(st); metas.append(m)
        print(f"[{name}] seed {s}: k={m['k']}/{X.shape[1]}  MINT-FS macro-F1="
              f"{[x for x in r if x['method']=='MINT-FS' and x['clf']=='HINT-Net'][0]['macro_f1']:.3f}  ({time.time()-t:.0f}s)", flush=True)
    out = dict(name=name, rows=allrows, sets=allsets, metas=metas, d=int(X.shape[1]), C=int(C),
               feat_names=list(feat_names) if feat_names is not None else None)
    if not QUICK: json.dump(out, open(path, "w"))
    return out

RESULTS = {}

## 2. Step 1 — Synthetic benchmarks (MC and BIN)

In [ ]:
if RUN_SYNTHETIC:
  SC = 6 if QUICK else 1          # QUICK: shrink the synthetic sets 6x for speed
  (X, y, info), Cmc = make_synthetic([n // SC for n in COUNTS_MC], binary=False, seed=0), 5
  RESULTS["Synthetic-MC"] = run_dataset("Synthetic-MC", X, y, Cmc, lam_sparse=3.0, info=info)
  (X, y, info), Cb = make_synthetic([n // SC for n in [13000, 780, 325, 130, 65]], binary=True, seed=1), 2
  RESULTS["Synthetic-BIN"] = run_dataset("Synthetic-BIN", X, y, Cb, lam_sparse=2.0, info=info)

## 3. Step 2 — Real datasets

Upload the three files to `/content/` (names below), then run the next two cells. Everything dataset-specific is already handled in
`prep_real.py`: identifiers, addresses, MACs, ports, timestamps, device ids and other label-leaking columns are removed, and for ECU-IoHT (which
only has 9 raw columns) per-packet features are derived from Protocol / Length / Info / Time.

In [ ]:
PATHS = {
    "WUSTL-EHMS-2020": "/content/wustl-ehms-2020_with_attacks_categories.csv",
    "WUSTL-HDRL-2024": "/content/wustl_hdrl_2024.csv",
    "ECU-IoHT":        "/content/ECU_IoHT.xlsx",      # .xlsx or .csv both work
}
# quick look: what each loader produces
for name in RUN:
    X, y, cols, cls = LOADERS[name](PATHS[name])
    print(f"{name}: X{X.shape}, classes={cls}, counts={np.bincount(y).tolist()}")
    print("  features:", cols)

In [ ]:
LAM = {"multi": 3.0, "bin": 2.0}
for name in RUN:
    X, y, cols, cls = LOADERS[name](PATHS[name])
    cnt = np.bincount(y); keep = np.where(cnt >= 20)[0]          # stratified 70/15/15 needs >= ~20 rows per class
    m = np.isin(y, keep); X, y = X[m], np.unique(y[m], return_inverse=True)[1]
    if MAX_ROWS and len(y) > MAX_ROWS:                           # stratified subsample, every row of the smallest classes is kept
        rng = np.random.RandomState(0); idx = []
        for c in np.unique(y):
            ii = np.where(y == c)[0]; n = min(len(ii), max(int(MAX_ROWS * len(ii) / len(y)), 1000))
            idx.append(rng.choice(ii, n, replace=False))
        idx = np.concatenate(idx); X, y = X[idx], y[idx]
    C = int(y.max() + 1)
    print(name, "X", X.shape, "classes", C, "counts", np.bincount(y).tolist(), flush=True)
    RESULTS[name] = run_dataset(name, X, y, C, lam_sparse=LAM["multi" if C > 2 else "bin"], feat_names=cols)

## 4. Step 3 — Tables (mean ± s.d. over the splits)

In [ ]:
METRICS = ["macro_f1", "gmean", "mcc", "pr_auc", "rare_recall"]
NICE = dict(macro_f1="Macro-F1", gmean="G-mean", mcc="MCC", pr_auc="PR-AUC", rare_recall="Rare recall")
ORDER = ["No-FS", "ANOVA-F", "MI", "RF-imp", "L1-LR", "mRMR", "MINT-FS"]

def pm(s, dec=3): return f"{s.mean():.{dec}f} ± {s.std(ddof=1):.{dec}f}" if len(s) > 1 else f"{s.mean():.{dec}f}"

def kuncheva_mean(R, method):
    sets = [s[method] for s in R["sets"] if method in s]; d = R["d"]
    v = [kuncheva(a, b, d) for a, b in itertools.combinations(sets, 2)]
    return (np.mean(v), np.std(v, ddof=1) if len(v) > 1 else 0.0) if v else (np.nan, np.nan)

def perf_table(R, clf="HINT-Net"):
    df = pd.DataFrame(R["rows"]); df = df[df.clf == clf]
    rows = []
    for m in ORDER:
        g = df[df.method == m]
        if g.empty: continue
        r = {"Method": m, "k": f"{g.k.mean():.1f}"}
        r.update({NICE[x]: pm(g[x]) for x in METRICS})
        if m != "No-FS":
            km, ks = kuncheva_mean(R, m); r["Kuncheva"] = f"{km:.2f} ± {ks:.2f}"
        else: r["Kuncheva"] = "–"
        r["Sel. time (s)"] = f"{g.sel_time.mean():.1f}"
        rows.append(r)
    return pd.DataFrame(rows)

def significance(R, clf="HINT-Net", metric="macro_f1"):
    from scipy.stats import wilcoxon
    df = pd.DataFrame(R["rows"]); df = df[df.clf == clf].pivot(index="seed", columns="method", values=metric)
    out = {}
    for m in ORDER:
        if m in ("MINT-FS",) or m not in df: continue
        diff = df["MINT-FS"] - df[m]
        try: p = wilcoxon(diff).pvalue if (diff != 0).any() and len(diff) >= 6 else np.nan
        except Exception: p = np.nan
        out[m] = dict(mean_diff=diff.mean(), wins=int((diff > 0).sum()), n=len(diff), p_wilcoxon=p)
    return pd.DataFrame(out).T

# pick up every dataset finished in this or an earlier session (results/*.json) so tables cover all datasets
for f in sorted(glob.glob("results/*.json")):
    R = json.load(open(f)); RESULTS.setdefault(R["name"], R)
TABLES = {}
for name, R in RESULTS.items():
    T = perf_table(R); TABLES[name] = T
    print("\n##", name, f"(d={R['d']}, classes={R['C']}, splits={len(R['sets'])})")
    display(T)
    print("MINT-FS vs each baseline (macro-F1, paired over splits):"); display(significance(R).round(4))

In [ ]:
# Summary across datasets (Macro-F1 and rare-class recall, MINT-FS vs No-FS vs best filter baseline)
summ = []
for name, R in RESULTS.items():
    df = pd.DataFrame(R["rows"]); df = df[df.clf == "HINT-Net"]
    base = [m for m in ORDER if m not in ("No-FS", "MINT-FS")]
    best = df[df.method.isin(base)].groupby("method").macro_f1.mean().idxmax()
    for lab, m in [("MINT-FS", "MINT-FS"), ("No-FS", "No-FS"), (f"best baseline ({best})", best)]:
        g = df[df.method == m]
        summ.append({"Dataset": name, "d": R["d"], "Method": lab, "k": f"{g.k.mean():.1f}",
                     "Macro-F1": pm(g.macro_f1), "Rare recall": pm(g.rare_recall), "MCC": pm(g.mcc)})
SUMMARY = pd.DataFrame(summ); display(SUMMARY)

# synthetic ground-truth recovery table
qrows = []
for name in [n for n in RESULTS if n.startswith("Synthetic")]:
    q = pd.DataFrame([m["quality"] for m in RESULTS[name]["metas"]])
    qrows.append({"Dataset": name, "k": f"{pm(pd.Series([m['k'] for m in RESULTS[name]['metas']]), 1)}",
                  "Informative kept": pm(q.n_info, 1), "Noise kept": pm(q.n_noise, 1),
                  "Redundant kept": pm(q.n_red, 1), "Synergy pair kept (splits)": f"{int(q.synergy.sum())}/{len(q)}"})
QUALITY = pd.DataFrame(qrows, columns=['Dataset','k','Informative kept','Noise kept','Redundant kept','Synergy pair kept (splits)']); display(QUALITY)

## 5. Export (CSV + LaTeX + Excel) and download

In [ ]:
assert not QUICK, "QUICK=True gives smoke-test numbers only - set QUICK=False and re-run before exporting"
import zipfile
with pd.ExcelWriter("results/paper_tables.xlsx") as xw:
    SUMMARY.to_excel(xw, "Summary", index=False); QUALITY.to_excel(xw, "Synthetic_recovery", index=False)
    for name, T in TABLES.items():
        T.to_excel(xw, name[:31], index=False); T.to_csv(f"results/table_{name}.csv", index=False)
        open(f"results/table_{name}.tex", "w").write(T.to_latex(index=False, escape=False))
        significance(RESULTS[name]).round(4).to_csv(f"results/signif_{name}.csv")
SUMMARY.to_csv("results/table_summary.csv", index=False); QUALITY.to_csv("results/table_synthetic_recovery.csv", index=False)
# selected feature names for the real datasets (most frequently selected over splits) -> use in the paper's discussion
sel_rows = []
for name, R in RESULTS.items():
    if R.get("feat_names"):
        cnt = pd.Series(itertools.chain.from_iterable(s["MINT-FS"] for s in R["sets"])).value_counts()
        for j, c in cnt.head(15).items(): sel_rows.append({"Dataset": name, "Feature": R["feat_names"][j], "Selected in splits": f"{c}/{len(R['sets'])}"})
if sel_rows: pd.DataFrame(sel_rows).to_csv("results/top_selected_features.csv", index=False); display(pd.DataFrame(sel_rows))
with zipfile.ZipFile("mintfs_results.zip", "w") as z:
    for f in glob.glob("results/*"): z.write(f)
print("wrote mintfs_results.zip", "(QUICK smoke-test numbers — do NOT use in the paper)" if QUICK else "")
try:
    from google.colab import files; files.download("mintfs_results.zip")
except Exception: pass